In [1]:
%run functions.ipynb

import os

os.makedirs("../submission", exist_ok=True)

df = load_data()
df.head()

,yt_true
date,
1946-01-01,890
1946-02-01,992
1946-03-01,979
1946-04-01,959
1946-05-01,1110


In [2]:
# Model 7: MLP autoregressive model over the differenced and scaled series
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

P = 13
df = remove_trend_and_cycle(df)

scaler = MinMaxScaler()
df["yt_true_d1d12_scaled"] = scaler.fit_transform(df[["yt_true_d1d12"]])
df = make_lagged_ts(df, P, "yt_true_d1d12_scaled")
x_columns = [f"lagged_{i}m" for i in range(1, P + 1)]

data = df.dropna()
X_complete, y_complete, X_train, y_train, X_test, y_test = train_test_split(
    data, x_columns, "yt_true_d1d12_scaled"
)

mlp = MLPRegressor(
    hidden_layer_sizes=(4,),
    activation="logistic",
    solver="lbfgs",
    max_iter=10000,
    random_state=42,
)
mlp.fit(X_train, y_train)

z_pred = pd.Series(np.nan, index=df.index)
z_pred[data.index] = scaler.inverse_transform(
    mlp.predict(X_complete).reshape(-1, 1)
).ravel()
df["yt_pred_mlp_d1d12"] = (
    z_pred + df.yt_true.shift(1) + df.yt_true.shift(12) - df.yt_true.shift(13)
)
plot_time_series(df[["yt_true", "yt_pred_mlp_d1d12"]])

<Figure size 1200x400 with 1 Axes>

In [3]:
metrics = compute_evaluation_metrics(df[["yt_true", "yt_pred_mlp_d1d12"]].dropna())
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_mlp_d1d12
0,MSE Train,66977.06
1,MSE Test,139232.21
2,MAE Train,189.92
3,MAE Test,268.96
